# 18. Plugins Semantic Kernel pour le test-time scaling

**Phase 6 (finale) de l'epic #2926** — suite de **NB-12** (moteurs) a **NB-17** (raisonnement natif).
Ce notebook **clot #2926** en exposant les moteurs de test-time scaling (BoN, Reflexion, routeur)
comme des **plugins [Semantic Kernel](https://github.com/microsoft/semantic-kernel)** (SK), faisant
le **pont avec la serie SemanticKernel** du depot.

## Pourquoi des plugins ?

Jusqu'ici (NB-12 a NB-17), les moteurs etaient des **fonctions Python** appelees a la main. Les
exposer en **plugins SK** les rend :
- **decouvrables** (le kernel connait leur schema : nom, description, parametres types),
- **composables** (un plugin peut en invoquer un autre via le kernel),
- **invocables par le LLM** (auto-function-calling : le modele choisit l'outil),
- **interoperables** avec le reste de l'ecosysteme SK (agents, process framework, MCP — serie
  SemanticKernel NB-03/06/08).

> Ide centrale : le **Kernel** est un **hub de composition**. Les moteurs de test-time scaling
> deviennent des **outils** que le kernel peut orchestrer — exactement la couche d'integration
> que SK apporte au-dessus des appels API bruts.

## Plan
1. Setup : Kernel + service SK (OpenAI/OpenRouter).
2. Plugin **BoN** (`best_of_n`).
3. Plugin **Reflexion** (`reflexion` avec verificateur).
4. Plugin **Routeur** (compose : invoque BoN ou Reflexion selon la strategie).
5. Composition via le kernel + pont auto-function-calling (exercice) vers la serie SemanticKernel.

## 0. Setup — Kernel Semantic Kernel + service OpenRouter

Semantic Kernel (Python, v1.x) se connecte a OpenAI. On le branche sur **OpenRouter** (comme
NB-12 a NB-17) via un client `AsyncOpenAI` passe au connecteur `OpenAIChatCompletion`.

In [1]:
%pip install -q semantic-kernel python-dotenv

import os, re, asyncio
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI
import semantic_kernel as sk
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIPromptExecutionSettings
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function, KernelArguments

_env_path = None
_current = Path.cwd()
for _i in range(10):
    if (_current / ".env").exists():
        _env_path = _current / ".env"; break
    if _current.name in ("GenAI", "MyIA.AI.Notebooks"):
        break
    _current = _current.parent
if _env_path is None:
    for _cand in (Path.cwd() / "MyIA.AI.Notebooks" / "GenAI" / ".env",
                  Path.cwd() / "GenAI" / ".env"):
        if _cand.exists():
            _env_path = _cand; break
if _env_path is not None:
    load_dotenv(_env_path); print(f".env charge depuis : {_env_path}")

print(f"semantic-kernel {sk.__version__}")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


.env charge depuis : C:\dev\CoursIA-2\MyIA.AI.Notebooks\GenAI\.env
semantic-kernel 1.41.3


In [2]:
FAST_MODEL = os.getenv("OPENAI_MODEL_FAST", "meta-llama/llama-3.3-70b-instruct")

def make_kernel():
    """Construit un Kernel SK branche sur OpenRouter (AsyncOpenAI custom)."""
    aoai = AsyncOpenAI(api_key=os.getenv("OPENROUTER_API_KEY"),
                       base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id=FAST_MODEL, async_client=aoai,
                                            service_id="or"))
    return kernel

async def llm(kernel, prompt, temperature=0.7, max_tokens=80):
    """Helper : un appel au service SK (cache la ceremony ChatHistory)."""
    svc = kernel.get_service("or")
    ch = ChatHistory(); ch.add_user_message(prompt)
    settings = OpenAIPromptExecutionSettings(max_tokens=max_tokens, temperature=temperature)
    r = await svc.get_chat_message_contents(chat_history=ch, settings=settings)
    return str(r[0])

kernel = make_kernel()
_ping = await llm(kernel, "Reponds uniquement par : OK", temperature=0.0, max_tokens=10)
print("Ping SK->OpenRouter :", repr(_ping[:30]))

Ping SK->OpenRouter : 'OK'


## 1. Le modele plugin de Semantic Kernel

Un **plugin** SK = une classe Python dont les methodes sont decorees par `@kernel_function`.
SK extrait le **schema** (nom, description, parametres + types) depuis la signature et la
docstring → le kernel sait les **decouvrir** et les **invoquer**. Les arguments passent par
`KernelArguments(**params)`.

> Pont vers la serie SemanticKernel : voir `MyIA.AI.Notebooks/GenAI/SemanticKernel/01-SemanticKernel-Intro.ipynb`
> (plugins de base) et `03-SemanticKernel-Agents.ipynb` (agents qui orchestrent des plugins).

On definit maintenant un **verificateur** reutilisable (extraction d'entier, herite de NB-16/17),
puis nos trois plugins.

In [3]:
def extraire_nombre(texte):
    m = re.findall(r'-?\d+', texte or "")
    return int(m[0]) if m else None

# Probleme de demo (reponse entiere verifiable) : sert aux demos BoN + Reflexion + routeur.
# On choisit un probleme "moyen" que le modele resout fiablement, pour demontrer le MECANISME
# plugin/composition SK (le but de ce notebook) — pas la robustesse du modele (cf NB-16/17).
DEMO = ("Un train part avec 45 passagers. 12 montent au suivant, 7 descendent. "
        "Combien reste-t-il de passagers ? Reponds uniquement par le nombre.")
DEMO_REPONSE = 50
print("Demo :", DEMO, "| attendu :", DEMO_REPONSE)

Demo : Un train part avec 45 passagers. 12 montent au suivant, 7 descendent. Combien reste-t-il de passagers ? Reponds uniquement par le nombre. | attendu : 50


## 2. Plugin BoN — `best_of_n`

Le moteur **Best-of-N** (NB-12/NB-16) en plugin : genere `n` echantillons, renvoie la liste des
reponses (l'appeleur peut ensuite les verifier / voter). Le plugin **depend du kernel** (pour le
service) — passe au constructeur.

In [4]:
class BoNPlugin:
    """Plugin SK : Best-of-N sampling (test-time scaling parallel)."""
    def __init__(self, kernel): self._k = kernel

    @kernel_function(name="best_of_n", description="Genere n reponses independantes a un prompt.")
    async def best_of_n(self, prompt: str, n: int = 3) -> str:
        reps = []
        for _ in range(n):
            reps.append((await llm(self._k, prompt, temperature=0.8)).strip())
        return " || ".join(reps)

kernel.add_plugin(BoNPlugin(kernel), plugin_name="bon")
print("Plugin 'bon' enregistre : fonction best_of_n(prompt: str, n: int = 3).")

Plugin 'bon' enregistre : fonction best_of_n(prompt: str, n: int = 3).


In [5]:
# Demo : invoque best_of_n via le kernel (pattern plug['fn'].invoke(kernel, args)).
_plug_bon = kernel.plugins["bon"]
_res_bon = await _plug_bon["best_of_n"].invoke(kernel, KernelArguments(prompt=DEMO, n=4))
print("BoN (n=4) reponses :", _res_bon)
print(f"Correctes (== {DEMO_REPONSE}) :", [r for r in str(_res_bon).split(' || ') if extraire_nombre(r) == DEMO_REPONSE])

BoN (n=4) reponses : 50 || 50 || 50 || 50
Correctes (== 50) : ['50', '50', '50', '50']


## 3. Plugin Reflexion — `reflexion` (avec verificateur)

Le moteur **Reflexion** (NB-12/NB-14) en plugin : K tours, verifie la reponse, renvoie le
diagnostic a l'LLM si rate. On injecte un **verificateur** (fonction `reponse -> bool`) au
constructeur — c'est le **pattern d'injection de dependance** de SK (le plugin ne sait pas
verifier, on lui fournit comment).

In [6]:
class ReflexionPlugin:
    """Plugin SK : Reflexion sequentielle avec verificateur injecte."""
    def __init__(self, kernel, verifier):
        self._k = kernel; self._verif = verifier

    @kernel_function(name="reflexion", description="K tours de Reflexion avec feedback du verificateur.")
    async def reflexion(self, prompt: str, k: int = 3) -> str:
        feedback = ""
        for tour in range(1, k + 1):
            if not feedback:
                invite = prompt
            else:
                invite = (f"{prompt}\n\nEssai precedent INCORRECT. Feedback : {feedback}\n"
                          f"Reessaie correctement. Reponds uniquement par le nombre.")
            rep = (await llm(self._k, invite, temperature=0.8)).strip()
            if self._verif(rep):
                return f"SUCCES tour {tour} : {rep}"
            feedback = f"ta reponse etait {extraire_nombre(rep)} (incorrect)"
        return f"ECHEC apres {k} tours : {rep}"

# Verificateur pour la demo (reponse attendue == DEMO_REPONSE, ici 50).
_verif_demo = lambda t: extraire_nombre(t) == DEMO_REPONSE
kernel.add_plugin(ReflexionPlugin(kernel, _verif_demo), plugin_name="reflexion")
print("Plugin 'reflexion' enregistre : fonction reflexion(prompt, k=3).")

Plugin 'reflexion' enregistre : fonction reflexion(prompt, k=3).


In [7]:
_plug_ref = kernel.plugins["reflexion"]
_res_ref = await _plug_ref["reflexion"].invoke(kernel, KernelArguments(prompt=DEMO, k=3))
print("Reflexion (K=3) :", _res_ref)

Reflexion (K=3) : SUCCES tour 1 : 50


## 4. Plugin Routeur — composition (un plugin qui en invoque d'autres)

Le **routeur** (NB-13) en plugin : selon une **strategie** ('bon' ou 'reflexion'), il invoque le
bon moteur **via le kernel**. C'est la **composition SK** : un plugin orchestre d'autres plugins,
le kernel reste le hub central.

In [8]:
class RouteurPlugin:
    """Plugin SK : routeur qui compose BoN/Reflexion via le kernel (pattern composition)."""
    def __init__(self, kernel): self._k = kernel

    @kernel_function(name="resoudre", description="Resout un probleme : strategie 'bon' ou 'reflexion'.")
    async def resoudre(self, prompt: str, strategie: str = "reflexion", n: int = 4, k: int = 3) -> str:
        if strategie == "bon":
            fn = self._k.plugins["bon"]["best_of_n"]
            r = await fn.invoke(self._k, KernelArguments(prompt=prompt, n=n))
            return f"[routeur->BoN n={n}] {r}"
        elif strategie == "reflexion":
            fn = self._k.plugins["reflexion"]["reflexion"]
            r = await fn.invoke(self._k, KernelArguments(prompt=prompt, k=k))
            return f"[routeur->Reflexion k={k}] {r}"
        return f"[routeur] strategie inconnue : {strategie}"

kernel.add_plugin(RouteurPlugin(kernel), plugin_name="routeur")
print("Plugin 'routeur' enregistre. Plugins disponibles :", list(kernel.plugins))

Plugin 'routeur' enregistre. Plugins disponibles : ['bon', 'reflexion', 'routeur']


In [9]:
# Demo composition : le routeur invoque Reflexion via le kernel.
_plug_rt = kernel.plugins["routeur"]
for strat in ("reflexion", "bon"):
    _r = await _plug_rt["resoudre"].invoke(kernel, KernelArguments(prompt=DEMO, strategie=strat))
    print(f"Routeur strategie={strat} -> {_r}")

Routeur strategie=reflexion -> [routeur->Reflexion k=3] SUCCES tour 1 : 50


Routeur strategie=bon -> [routeur->BoN n=4] 50 || 50 || 50 || 50


## 5. Composition via le Kernel — au-dela de l'invocation manuelle

On a invoque les plugins **manuellement** (`plug['fn'].invoke(kernel, args)`). La puissance de SK
est l'**auto-function-calling** : on enregistre les plugins, et le **LLM choisit lui-meme** quel
plugin appeler (avec quels arguments) en voyant leur schema — c'est le routeur **agentique** natif
de SK (cf serie SemanticKernel `03-Agents` et NB-13 routeur).

> **Pont** : l'auto-function-calling SK generalise le routeur ad-hoc de NB-13. La serie
> SemanticKernel (NB-01 intro, NB-03 agents, NB-06 process framework, NB-08 MCP) couvre en
> profondeur l'orchestration de plugins ; ce notebook est le **point d'entree** depuis la serie
> test-time scaling.

L'auto-function-calling (boucle tool-use geree par SK) est laisse en **exercice 1** — il demande
d'activer `tool_call_behavior` sur les settings et de gerer la boucle, ce qui est le sujet des
notebooks dedies de la serie SemanticKernel.

## 6. Limites honnetes (G.2) et synthesis de l'epic #2926

- **Plugins = enveloppe** : la logique des moteurs (BoN/Reflexion/routeur) est **heritee de
  NB-12 a NB-17** ; ce notebook ajoute la **couche d'integration SK** (decouverte, composition,
  interop), pas de nouvel algorithme. C'est honnête : la valeur ici est l'**architecture**, pas
  une nouvelle technique de scaling.
- **Routeur manuel** : la composition est explicite (`strategie='bon'`). L'auto-routing par le
  LLM (exercice 1) est plus puissant mais token-ponderereux et instable — laissé en exercice.
- **Une seule famille de plugins** (scaling) : SK brille avec des plugins **heterogenes**
  (recherche, code, BDD) ; la serie SemanticKernel montre cela en profondeur.

**Synthesis de l'epic #2926** (test-time scaling / ICR) — 6 phases :
- **NB-13** routeur agentique (Ph1) — **NB-14** memoire persistante (Ph2) — **NB-15** ToT sur CSP (Ph3) —
  **NB-16** scaling pass@k / Snell (Ph4) — **NB-17** raisonnement natif vs scaling (Ph5) —
  **NB-18** plugins SK (Ph6, ce notebook).

## 7. Travaux pratiques

Les exercices sont a completer (convention C.1 : pas d'erreur volontaire).

### Exercice 1 : auto-function-calling (le LLM choisit le plugin)

Active l'**auto-function-calling** de SK : enregistre `bon` et `reflexion`, donne un prompt au
LLM, et laisse SK appeler le bon plugin automatiquement. Compare le routing natif SK au routeur
ad-hoc de NB-13.

**Indice :** OpenAIPromptExecutionSettings(tool_call_behavior=...) + kernel.invoke avec auto-call.
Cf serie SemanticKernel 03-Agents et la doc SK "function calling".

In [10]:
async def auto_route(kernel, prompt):
    """Exercice 1 : auto-function-calling — le LLM choisit entre bon et reflexion."""
    # TODO etudiant : activer tool_call_behavior, invoquer avec auto-call, retourner la trace.
    return None

print(f"Exercice 1 - auto-function-calling : {'implemente' if False else 'a completer'}")

Exercice 1 - auto-function-calling : a completer


### Exercice 2 : plugin Memory (pont NB-14)

Reprend la **memoire vectorielle** de NB-14 et expose-la comme plugin SK : un plugin
`memoire.retroceder(query, k)` + `memoire.ajouter(lesson)`. Le routeur peut alors injecter les
lecons passees avant d'appeler Reflexion.

**Indice :** wrap la MemoireVectorielle de NB-14 dans une classe avec @kernel_function sur
retroceder/ajouter ; le routeur appelle memoire.retroceder puis reflexion.

In [11]:
class MemoirePlugin:
    """Exercice 2 : plugin SK wrappant la MemoireVectorielle de NB-14."""
    # TODO etudiant : __init__(kernel, memoire) + @kernel_function retroceder(query, k) + ajouter(lesson).
    pass

print(f"Exercice 2 - plugin Memoire : {'implemente' if False else 'a completer'}")

Exercice 2 - plugin Memoire : a completer


### Exercice 3 (avance) : orchestrer via le Process Framework (serie SemanticKernel NB-06)

Encode le flux "routeur -> moteur -> verificateur -> memoire" comme un **Process Framework** SK
(etats + transitions) plutot que des appels de plugins. Pont direct vers
`SemanticKernel/06-SemanticKernel-ProcessFramework.ipynb`.

**Indice :** kernel.add_process(...) avec des steps (BoNStep, ReflexionStep, MemoryStep) et des
edges ; le process orchestre les plugins sans orchestration manuelle.

In [12]:
def build_scaling_process(kernel):
    """Exercice 3 : encode le flux test-time scaling comme un Process Framework SK."""
    # TODO etudiant : definir des ProcessStep + edges, retourner le process pret a demarrer.
    return None

print(f"Exercice 3 - Process Framework : {'implemente' if False else 'a completer'}")

Exercice 3 - Process Framework : a completer


## 8. Conclusion

On a expos les moteurs de test-time scaling (BoN, Reflexion, routeur — herites de NB-12 a NB-17)
comme des **plugins Semantic Kernel**, demontrant le **modele de composition** de SK : plugins
decouvrables, invocables via le kernel, et composables (le routeur invoque BoN/Reflexion via le
kernel). Ce notebook est le **point d'entree** depuis la serie test-time scaling vers la serie
SemanticKernel (plugins, agents, process framework, MCP).

**#2926 cloture** : 6 phases livrees — routeur agentique (NB-13), memoire persistante (NB-14),
ToT sur CSP (NB-15), scaling pass@k / Snell (NB-16), raisonnement natif vs scaling (NB-17),
plugins SK (NB-18, ce notebook).

**References :** Semantic Kernel (Microsoft) — serie `SemanticKernel/` du depot ; Snell et al. 2024
(test-time scaling) ; Yao et al. 2023 (Tree of Thoughts) ; NB-12 a NB-17 de cette serie.